In [1]:
import matplotlib.pyplot as plt
from matplotlib.transforms import blended_transform_factory
from pathlib import Path
from __future__ import annotations


In [ ]:
BASE_DIR = Path.cwd()
NETWORK_DIR = BASE_DIR / "network_diagrams"
GROWTH_PLOT_DIR = BASE_DIR / "growth_growth_plots"
PLOT_FIG_TYPE = "png"  # can be "png" or "svg"
SAVE_MODE = "both"

figures = {
    "main"                      : {
        "simulations": ["1_3_parallel", "2_1_crossfeed", "3_2_competition", "4_3_coupling"],
        "letter_x_col0": 0.07,
        "letter_x_col1": 0.52,
        "outfile": BASE_DIR / "paper_figs" / f"main_figure.{PLOT_FIG_TYPE}"
    },
    "non_interacting_supp"      : {
        "simulations": ["1_1_single", "1_2_single"],
        "letter_x_col0": 0.1,
        "letter_x_col1": 0.47,
        "outfile": BASE_DIR / "supplement" / f"non_interacting.{PLOT_FIG_TYPE}"
    },
    "crossfeed_supp"            : {
        "simulations": ["2_2_layered", "2_3_superfluous_crossfeed"],
        "letter_x_col0": 0.05,
        "letter_x_col1": 0.57,
        "outfile": BASE_DIR / "supplement" / f"crossfeed.{PLOT_FIG_TYPE}"
    },
    "competition_supp"          : {
        "simulations": ["3_1_competition", "3_3_competition", "3_4_competition", "3_5_competition"],
        "letter_x_col0": 0.07,
        "letter_x_col1": 0.47,
        "outfile": BASE_DIR / "supplement" / f"competition.{PLOT_FIG_TYPE}"
    },
    "crossfeed_competition_supp": {
        "simulations": ["4_1_crossfeed_competition", "4_2_coupling", "4_4_dynamical"],
        "letter_x_col0": 0.05,
        "letter_x_col1": 0.52,
        "outfile": BASE_DIR / "supplement" / f"crossfeed_competition.{PLOT_FIG_TYPE}"
    },
    "bifurcation_supp": {
        "simulations": ["4_3_coupling", "4_4_dynamical"],
        "letter_x_col0": 0.05,
        "letter_x_col1": 0.52,
        "outfile": BASE_DIR / "supplement" / f"bifurcation.{PLOT_FIG_TYPE}"
    }
}



for figure in figures.keys():
    sim_list = figures[figure]["simulations"]

    networks = []
    plots = []
    width_scales = []
    max_growth_w = 0
    for sim in sim_list:
        # lead network diagram
        network_file = NETWORK_DIR / f"{sim}_diagram.png"
        if figure == "bifurcation_supp":
            network_file = NETWORK_DIR / f"{sim}_bifurc_model_diagram.png"
        network = plt.imread(network_file)
        networks.append(network)

        # load growth-growth plot
        growth_plot_file = GROWTH_PLOT_DIR / f"{sim}.png"
        if figure == "bifurcation_supp":
            growth_plot_file = GROWTH_PLOT_DIR / f"{sim}_bifurcation_plot.png"
        growth = plt.imread(growth_plot_file)
        plots.append(growth)

        network_h, network_w, _ = network.shape
        growth_h, growth_w, _ = growth.shape

        network_w_scaled = network_w * (growth_h / network_h)
        width_scales.append(network_w_scaled)
        max_growth_w = max(max_growth_w, growth_w)
for figure in figures.keys():
    sim_list = figures[figure]["simulations"]
    outfile = figures[figure]["outfile"]

    networks = []
    plots = []
    width_scales = []
    max_growth_w = 0
    
    for sim in sim_list:
        # lead network diagram
        network_file = NETWORK_DIR / f"{sim}_diagram.png"
        network = plt.imread(network_file)
        networks.append(network)

        # load growth-growth plot
        growth_plot_file = GROWTH_PLOT_DIR / f"{sim}.png"
        growth = plt.imread(growth_plot_file)
        plots.append(growth)

        network_h, network_w, _ = network.shape
        growth_h, growth_w, _ = growth.shape

        network_w_scaled = network_w * (growth_h / network_h)
        width_scales.append(network_w_scaled)
        max_growth_w = max(max_growth_w, growth_w)

    num_rows = len(sim_list)
    fig_height = max(4, 2 * num_rows) 
    
    # Save Full Combined Figure
    if SAVE_MODE in ["full", "both"]:
        # create figure with two subplots per row - ensuring both images are the same height
        fig, axs = plt.subplots(
            nrows=num_rows, 
            ncols=2, 
            figsize=(8, fig_height),
            gridspec_kw={'width_ratios': [max(width_scales), max_growth_w]},
            squeeze=False 
        )

        for sim_idx, sim in enumerate(sim_list):
            network = networks[sim_idx]
            growth = plots[sim_idx]

            axs[sim_idx, 0].axis('off')
            axs[sim_idx, 0].imshow(network, interpolation='lanczos')

            axs[sim_idx, 1].axis('off')
            axs[sim_idx, 1].imshow(growth, interpolation='lanczos')

            # Note: tight_layout is best called once outside the loop for the full figure
            plt.tight_layout()

            # transforms to make all letters horizontally aligned
            trans0 = blended_transform_factory(fig.transFigure, axs[sim_idx, 0].transAxes)
            trans1 = blended_transform_factory(fig.transFigure, axs[sim_idx, 1].transAxes)
            x_0 = figures[figure].get("letter_x_col0", 0.05)
            x_1 = figures[figure].get("letter_x_col1", 0.5)

            axs[sim_idx, 0].text(x_0, 1.05, f"{chr(65 + 2 * sim_idx)}", transform=trans0, 
                        fontsize=12, va='top', ha='center')
            axs[sim_idx, 1].text(x_1, 1.05, f"{chr(66 + 2 * sim_idx)}", transform=trans1, 
                        fontsize=12, va='top', ha='left')

        plt.savefig(outfile, bbox_inches='tight', dpi=600)
        plt.close(fig)

    # Save Individual Rows
    if SAVE_MODE in ["rows", "both"]:
        row_height = fig_height / num_rows

        for sim_idx, sim in enumerate(sim_list):
            fig, axs = plt.subplots(
                nrows=1, 
                ncols=2, 
                figsize=(8, row_height),
                # Using the exact same gridspec scales guarantees rows match if restacked later
                gridspec_kw={'width_ratios': [max(width_scales), max_growth_w]},
                squeeze=False 
            )

            network = networks[sim_idx]
            growth = plots[sim_idx]

            axs[0, 0].axis('off')
            axs[0, 0].imshow(network, interpolation='lanczos')

            axs[0, 1].axis('off')
            axs[0, 1].imshow(growth, interpolation='lanczos')

            plt.tight_layout()

            trans0 = blended_transform_factory(fig.transFigure, axs[0, 0].transAxes)
            trans1 = blended_transform_factory(fig.transFigure, axs[0, 1].transAxes)
            x_0 = figures[figure].get("letter_x_col0", 0.05)
            x_1 = figures[figure].get("letter_x_col1", 0.5)

            # Keep the continuous lettering (A, B, C, D, etc.) even when saved separately
            axs[0, 0].text(x_0, 1.05, f"{chr(65 + 2 * sim_idx)}", transform=trans0, 
                        fontsize=12, va='top', ha='center')
            axs[0, 1].text(x_1, 1.05, f"{chr(66 + 2 * sim_idx)}", transform=trans1, 
                        fontsize=12, va='top', ha='left')

            # Save with suffix e.g., main_figure_row_1.png
            row_outfile = outfile.parent / f"{outfile.stem}_row_{sim_idx + 1}{outfile.suffix}"
            plt.savefig(row_outfile, bbox_inches='tight', dpi=600)
            plt.close(fig)